# HSI Depth Of Field Calibration


## Import Libraries


In [ ]:
import os
import sys
import math
import numpy as np
import pandas as pd
import spectral
from IPython.core import ultratb
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from shapely.geometry import Polygon, Point
from PIL import Image
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from skimage.draw import polygon

# Enable support for ENVI header files with non-lowercase parameter names
spectral.settings.envi_support_nonlowercase_params = True

## Read HSI


In [ ]:
def read_hsi(hdr_file, start_index, end_index):
    """
    Read a hyperspectral image from band index start_index up to exd_index,
    """

    # load the image object (to get metadata) and memmap the data
    img_obj = spectral.open_image(hdr_file)

    # slice out only bands start_index through end_index-1
    subcube = img_obj.read_bands(range(start_index, end_index))

    # grab the matching wavelengths
    all_wls = img_obj.metadata["wavelength"]
    wavelengths = [float(w) for w in all_wls[start_index:end_index]]

    return subcube, wavelengths

## Bands Visualization


In [ ]:
def plot_bands(image, wavelengths, n=1):
    """
    Plots each band (or every n bands) as a grayscale image.
    """
    # Get number of bands
    num_bands = image.shape[-1]

    # Select bands based on step size
    bands_to_plot = list(range(0, num_bands, n))

    # Determine plot grid
    num_plots = len(bands_to_plot)
    cols = min(5, num_plots)  # Set a max of 5 columns
    rows = (num_plots // cols) + (num_plots % cols > 0)

    # Create subplots
    fig, axes = plt.subplots(rows, cols, figsize=(15, 3 * rows))
    axes = np.atleast_2d(axes)  # Ensure 2D array for indexing

    # Plot each selected band
    for idx, band in enumerate(bands_to_plot):
        row, col = divmod(idx, cols)
        ax = axes[row, col]
        ax.imshow(image[:, :, band], cmap="gray")
        ax.set_title(f"{wavelengths[band]} nm")
        ax.axis("off")

    # Remove empty subplots if any
    for idx in range(num_plots, rows * cols):
        row, col = divmod(idx, cols)
        fig.delaxes(axes[row, col])

    plt.tight_layout()
    plt.show()

## Average Bands


In [ ]:
def average_bands(image, save_path=None):
    """
    Reads a hyperspectral image and computes the average grayscale image
    """
    # Compute the average grayscale image across the selected bands
    avg_image = np.mean(image, axis=2)

    # Display the averaged grayscale image
    fig = plt.gcf()
    fig.set_dpi(300)
    plt.imshow(avg_image, cmap="gray")
    plt.title(f"Average Image")
    plt.axis("off")
    
    # Save the figure if save_path is provided
    if save_path:
        # Save only the image without title or margins
        plt.savefig(save_path, dpi=300, bbox_inches='tight', pad_inches=0)
        print(f"Figure saved to: {save_path}")
    
    plt.show()

    return avg_image

## Pattern Extraction


In [ ]:
def pattern_extraction(xml_file, image, subset=None, save_path=None):
    """
    Extracts the ROI polygon from a grayscale image based on the XML,
    then (optionally) crops vertically to subset = [start_row, end_row].
    """
    # parse XML & get polygon verts
    tree = ET.parse(xml_file)
    root = tree.getroot()
    region = root.find(".//Region")
    roi_name = region.get("name") if region is not None else "ROI"

    coords_text = root.find(".//Coordinates").text.strip()
    vals = np.fromstring(coords_text, sep=" ")
    verts = vals.reshape(-1, 2).round().astype(int)
    xs, ys = verts[:, 0], verts[:, 1]

    # rasterize mask & crop to bounding box
    mask = np.zeros(image.shape, dtype=bool)
    rr, cc = polygon(ys, xs, image.shape)
    mask[rr, cc] = True

    minr, maxr = rr.min(), rr.max() + 1
    minc, maxc = cc.min(), cc.max() + 1
    cropped = image[minr:maxr, minc:maxc]
    cropped_mask = mask[minr:maxr, minc:maxc]
    roi = np.where(cropped_mask, cropped, 0)

    # optional vertical subset
    if subset and len(subset) == 2:
        s, e = subset
        # clamp to valid range
        s = max(0, int(round(s)))
        e = min(roi.shape[0], int(round(e)))
        roi = roi[s:e, :]

    # display
    plt.figure(figsize=(5, 5))
    plt.imshow(roi, cmap="gray", vmin=0, vmax=roi.max())
    plt.title(roi_name + (f"  (rows {subset[0]}–{subset[1]})" if subset else ""))
    plt.axis("off")
    
    # Save the figure if save_path is provided
    if save_path:
        plt.savefig(save_path, format="svg", bbox_inches='tight', pad_inches=0)
        print(f"Pattern saved to: {save_path}")
    
    plt.show()

    return roi

In [ ]:
def average_pattern_intensity(roi_cropped, output_csv="roi_intensity.csv"):
    """
    Calculates the mean intensity along the x-axis of the extracted ROI
    """
    # Compute mean intensity along x-axis
    mean_intensity = np.nanmean(roi_cropped, axis=1)
    y_pixels = np.arange(len(mean_intensity))

    # Save to CSV
    # df = pd.DataFrame({"Pixel_Y": y_pixels, "Mean_Intensity": mean_intensity})
    # df.to_csv(output_csv, index=False)
    # print(f"Saved intensity profile to {output_csv}")

    # Plot intensity profile along y-axis
    # plt.figure(figsize=(8, 5))
    # plt.plot(y_pixels, mean_intensity)
    # plt.xlabel("Pixel Location")
    # plt.ylabel("Mean Intensity")
    # plt.title(f"Intensity Profile Along Y-Axis")
    # plt.grid(True)
    # plt.show()

    return mean_intensity

## Contrast Transfer Function (CTF)


In [ ]:
def compute_ctf(intensity_profile, ctf_threshold=0.2, save_path=None):
    """
    Finds all peak–valley–peak triplets, filters for CTF >= threshold
    """
    # 1) Smooth
    smoothed = gaussian_filter1d(intensity_profile, sigma=1.1)

    # 2) Find peaks & valleys
    peaks, _ = find_peaks(smoothed, distance=5)
    valleys, _ = find_peaks(-smoothed, distance=5)

    # 3) Build triplets
    triplets = []
    for i in range(len(peaks) - 1):
        p1, p2 = peaks[i], peaks[i + 1]
        vs = valleys[(valleys > p1) & (valleys < p2)]
        if vs.size:
            v = vs[np.argmin(np.abs(vs - p1))]
            triplets.append((p1, v, p2))

    # 4) Filter by CTF
    filtered = []
    for p1, v, p2 in triplets:
        Imax, Imin = smoothed[p1], smoothed[v]
        ctf = (Imax - Imin) / (Imax + Imin)
        if ctf >= ctf_threshold:
            filtered.append((p1, v, p2, ctf))

    # 5) Early exit
    if not filtered:
        print("No CTF‑qualified peak–valley–peak triplets found.")
        return np.array([]), None, None, [], []

    # Count pairs
    valid_peaks = {p for p, _, _, _ in filtered} | {p for _, _, p, _ in filtered}
    valid_vals = {v for _, v, _, _ in filtered}

    # 6) Extract region from ORIGINAL profile
    start_idx = filtered[0][0]
    end_idx = filtered[-1][0]
    region = intensity_profile[start_idx : end_idx + 1]

    triplet_pairs = [(p1, v, p2) for p1, v, p2, _ in filtered]
    ctf_vals = [(p1, ctf) for p1, _, _, ctf in filtered]

    # 7) Plot
    y = np.arange(len(smoothed))
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(y, smoothed, label="Mean Intensity")
    ax.scatter(
        list(valid_peaks),
        smoothed[list(valid_peaks)],
        marker="o",
        label="Peaks",
        color="red",
    )
    ax.scatter(
        list(valid_vals),
        smoothed[list(valid_vals)],
        marker="x",
        label="Valleys",
        color="green",
    )

    # annotate first & last CTF values and vertical lines
    first, last = filtered[0], filtered[-1]
    for p1, _, _, ctf in (first, last):
        ax.annotate(
            f"{ctf:.2f}",
            (p1, smoothed[p1]),
            textcoords="offset points",
            xytext=(0, 8),
            ha="center",
            color="red",
            fontsize=10,
        )
    ax.axvline(start_idx, color="orange", linestyle="--", linewidth=1)
    ax.axvline(end_idx, color="orange", linestyle="--", linewidth=1)
    ax.text(
        start_idx,
        0.01,
        start_idx,
        transform=ax.get_xaxis_transform(),
        va="bottom",
        ha="right",
        color="orange",
        fontsize=9,
    )
    ax.text(
        end_idx,
        0.01,
        end_idx,
        transform=ax.get_xaxis_transform(),
        va="bottom",
        ha="left",
        color="orange",
        fontsize=9,
    )

    ax.text(
        0.02,
        0.95,
        f"LP#: {len(filtered)}",
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=10,
        bbox=dict(facecolor="white", alpha=0.7, edgecolor="gray"),
    )

    plt.xlabel("Pixel Location")
    plt.ylabel("Intensity")
    plt.legend(
        loc="upper right",
    )
    plt.grid(True)
    
    # Save the figure if save_path is provided
    if save_path:
        plt.savefig(save_path, format="svg", dpi=300, bbox_inches='tight')
        print(f"CTF plot saved to: {save_path}")
    
    plt.show()

    return region, start_idx, end_idx, triplet_pairs, ctf_vals

## Main


In [ ]:
# Set the .hdr file path
hdr_file = "/mnt/c/data/hyperbird_calibration/dof_processed/168-target/168-target_ffc.hdr"

# Read the .hdr file
image, wavelengths = read_hsi(hdr_file=hdr_file, start_index=50, end_index=150)

In [ ]:
import numpy as np
import imageio.v3 as iio
import matplotlib.pyplot as plt

# Read the image (replace with your image path)
avg_image = iio.imread("../data/f11.png", mode='L')  # e.g., shape (H, W, 3) or (H, W, 4)

plt.axis("off")
plt.title("Grayscale Image")
plt.imshow(avg_image, cmap='gray')
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Plot the band visualization
# plot_bands(image=image, wavelengths=wavelengths)

# Average the selected bands for clearer visualization of the patterns
avg_image = average_bands(image=image, save_path="../figures/average_bands.png")

In [ ]:
# Extract and analyze the resolution pattern

roi_xml_path = "/mnt/c/data/hyperbird_calibration/dof_processed/15lp.xml"
roi_cropped = pattern_extraction(roi_xml_path, avg_image)
mean_intensity = average_pattern_intensity(roi_cropped)

In [ ]:
# Determine the peaks and valleys and calculate the CTF values
_, start_idx, end_idx, _, _ = compute_ctf(mean_intensity, ctf_threshold=0.10, save_path="../figures/ctf_analysis.svg")
label_roi_xml_path = (
    "/mnt/c/data/hyperbird_calibration/dof_processed//label.xml"
)
roi_cropped = pattern_extraction(label_roi_xml_path, avg_image, [start_idx, end_idx])